# Setup Verification — Week 5: Fine-Tuning LLMs

**Learning Objectives:**
- Verify all required packages are installed for your chosen training path
- Confirm environment variables are configured correctly
- Validate API connectivity to Claude and Ollama

**Estimated Time:** 10 minutes

**Path Indicator:** Run all cells — this notebook is path-agnostic and checks everything.

In [1]:
import sys
import importlib

sys.path.insert(0, "..")

import src
importlib.reload(src)

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

from src.cost_tracker import CostTracker
tracker = CostTracker()

print("Core imports OK")
print(f"Python: {sys.version}")

Core imports OK
Python: 3.12.13 (main, Mar 25 2026, 03:16:06) [Clang 22.1.1 ]


## Part 1: Package Check

We check four groups of packages. A ✓ means importable; a ✗ means you need to install it.
Some packages are path-specific or platform-specific — those are noted inline.

In [2]:
import platform
import subprocess

def check_import(module_name, display_name=None):
    name = display_name or module_name
    try:
        importlib.import_module(module_name)
        print(f"  ✓ {name}")
        return True
    except ImportError as e:
        print(f"  ✗ {name}  ({e})")
        return False

is_apple_silicon = (platform.system() == "Darwin" and platform.machine() == "arm64")

print("=" * 50)
print("Group 1: Core packages")
print("=" * 50)
check_import("anthropic")
check_import("ollama")
check_import("requests")
check_import("dotenv", "python-dotenv")

print()
print("=" * 50)
print("Group 2: Training core (Path B)")
print("=" * 50)
check_import("transformers")
check_import("peft")
check_import("trl")
check_import("datasets")
check_import("accelerate")

print()
print("=" * 50)
print(f"Group 3: MLX (Path A) — Apple Silicon: {is_apple_silicon}")
print("=" * 50)
if is_apple_silicon:
    check_import("mlx")
    check_import("mlx_lm")
else:
    print("  — Skipped (not Apple Silicon)")

print()
print("=" * 50)
print("Group 4: QLoRA / bitsandbytes (Path B, Linux GPU only)")
print("=" * 50)
if platform.system() == "Darwin":
    print("  — Skipped on Mac (bitsandbytes requires Linux + CUDA)")
else:
    check_import("bitsandbytes")

print()
print("=" * 50)
print("Group 5: Serving")
print("=" * 50)
check_import("llama_cpp", "llama-cpp-python (optional)")

try:
    result = subprocess.run(["ollama", "--version"], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print(f"  ✓ ollama CLI: {result.stdout.strip()}")
    else:
        print(f"  ✗ ollama CLI: {result.stderr.strip()}")
except FileNotFoundError:
    print("  ✗ ollama CLI: not found in PATH")
except subprocess.TimeoutExpired:
    print("  ✗ ollama CLI: timed out")

Group 1: Core packages
  ✓ anthropic
  ✓ ollama
  ✓ requests
  ✓ python-dotenv

Group 2: Training core (Path B)
  ✓ transformers
  ✓ peft
  ✓ trl
  ✓ datasets
  ✓ accelerate

Group 3: MLX (Path A) — Apple Silicon: True
  ✓ mlx
  ✓ mlx_lm

Group 4: QLoRA / bitsandbytes (Path B, Linux GPU only)
  — Skipped on Mac (bitsandbytes requires Linux + CUDA)

Group 5: Serving
  ✗ llama-cpp-python (optional)  (No module named 'llama_cpp')
  ✓ ollama CLI: ollama version is 0.20.2


## Part 2: Environment Variables Check

Two variables must be set:
- `ANTHROPIC_API_KEY` — for Claude API calls
- `OLLAMA_HOST` — for local Ollama inference (defaults to `http://localhost:11434` if unset)

In [3]:
import os

def check_env(var, sensitive=False, default=None):
    val = os.environ.get(var)
    if val:
        display = f"{val[:8]}..." if sensitive else val
        print(f"  ✓ {var} = {display}")
        return True
    elif default:
        print(f"  ~ {var} not set — will use default: {default}")
        return True
    else:
        print(f"  ✗ {var} not set")
        return False

print("Environment variables:")
api_ok  = check_env("ANTHROPIC_API_KEY", sensitive=True)
host_ok = check_env("OLLAMA_HOST", default="http://localhost:11434")

if not api_ok:
    print()
    print("ACTION REQUIRED: Set ANTHROPIC_API_KEY in your .env file")
    print("  echo 'ANTHROPIC_API_KEY=sk-...' >> ../.env")

Environment variables:
  ✓ ANTHROPIC_API_KEY = sk-ant-a...
  ✓ OLLAMA_HOST = http://localhost:11434


## Part 3: Quick API Test

We make a minimal call to each available API. This confirms credentials work and latency is acceptable.

In [4]:
from src.llm_client import LLMClient
from src.config import CLAUDE_MODEL, OLLAMA_MODEL

# Path A = Claude, Path B = Ollama
llm_a = LLMClient(path="A")   # Claude
llm_b = LLMClient(path="B")   # Ollama (needs `ollama serve`)

# Test Claude
print("--- Claude API test ---")
try:
    resp = llm_a.generate(
        prompt="Say hello in one sentence.",
        model=CLAUDE_MODEL,
        max_tokens=64
    )
    tracker.add_call(resp)
    response = resp.get("content", resp.get("error", "no content"))
    print(f"Claude says: {response}")
except Exception as e:
    response = None
    print(f"Claude API error: {e}")

print()

# Test Ollama
print("--- Ollama API test ---")
try:
    resp_b = llm_b.generate(
        prompt="Say hello in one sentence.",
        model=OLLAMA_MODEL,
        max_tokens=64
    )
    tracker.add_call(resp_b)
    print(f"Ollama says: {resp_b.get('content', resp_b.get('error', 'no content'))}")
except Exception as e:
    print(f"Ollama API error (is `ollama serve` running?): {e}")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
✓ Ollama client initialized
  Available models: ['llama3.2-vision:latest', 'qwen3.5:27b', 'qwen3.5:9b']
  Default model: qwen3.5:27b
--- Claude API test ---
Claude says: Hello! I hope you're having a wonderful day! 😊

--- Ollama API test ---
Ollama says: 


## Summary

In [5]:
from src.utils import append_to_reflection

tracker.report()

if 'response' in dir():
    append_to_reflection(
        notebook="00_setup_verification",
        section_title="Setup Verification",
        reflection_content="Setup verification complete. All package groups checked, env vars confirmed, Claude and Ollama APIs tested."
    )

API COST REPORT
Total API calls:     2
Total input tokens:  29
Total output tokens: 273
Total cost:          $0.0003

Last 2 calls:
  1. [16:45:41] sonnet -- 13in/17out -- $0.0003
  2. [16:46:47] qwen3.5:27b -- 16in/256out -- $0.0000
